In [3]:
# prompt: use OpenAI GPT4o to extract first columns' paper title. My prompt: Please extract paper title from this sentence.

!pip uninstall -y openai
!pip install --upgrade openai

import openai
print(openai.__version__)
import pandas as pd

# Assuming 'senior_author' DataFrame is loaded and contains a column named 'Paper Title'

# Set your OpenAI API key
openai.# Replace with your actual API key

Found existing installation: openai 1.69.0
Uninstalling openai-1.69.0:
  Successfully uninstalled openai-1.69.0
     |████████████████████████████████| 599 kB 6.5 MB/s eta 0:00:01
1.70.0


In [20]:
# Create output folder
input_dir = "MedBullet_Reasoning_Maps"
os.makedirs(input_dir, exist_ok=True)

# Write empty JSON files for each row
for idx, _ in df_op4.iterrows():
    filename = f"medbullet_q{idx+1:03d}.json"
    filepath = os.path.join(input_dir, filename)
    
    # Save an empty JSON object
    with open(filepath, 'w') as f:
        json.dump({}, f, indent=2)

print(f"✅ Created {len(df_op4)} empty JSON files in '{input_dir}' folder.")

✅ Created 298 empty JSON files in 'MedBullet_Reasoning_Maps' folder.


In [21]:
import pandas as pd
import os
import json

# Load the CSV file
df_op4 = pd.read_csv("medbullets_op4.csv")

# Remove duplicate questions
num_duplicates = df_op4.duplicated(subset=['question']).sum()
df_op4 = df_op4.drop_duplicates(subset=['question'], keep='first')
print(f"Number of duplicate rows removed: {num_duplicates}")
print(f"Remaining rows: {len(df_op4)}")

# Create output folder
input_dir = "MedBullet_Reasoning_Maps"
os.makedirs(input_dir, exist_ok=True)

Number of duplicate rows removed: 10
Remaining rows: 298


In [35]:
import openai
import json

def generate_reasoning_tree_json(question, options, model="gpt-4o"):
    """
    Given a clinical question and 4 options, returns a structured reasoning tree JSON from GPT-4o.
    """

    prompt = f"""
You are a clinical reasoning assistant.

A user is asking you to analyze the following USMLE-style clinical question and return a structured JSON file that captures the diagnostic reasoning process.

---

**Question:**
{question}

**Options:**
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

---

**Instructions:**

Your output must be a valid JSON object that represents a diagnostic reasoning tree. 

For each diagnostic option, analyze its strengths and weaknesses **using only the specific information provided in the question prompt**. Do **not** answer based on general knowledge alone. Reference key clinical details from the vignette, including:

- vital signs (e.g., fever, hypotension, tachycardia),
- past medical history (e.g., IV drug use, cancer, immunosuppression),
- physical exam findings (e.g., murmurs, rashes, neurological signs),
- presenting symptoms or contextual clues (e.g., confusion, diarrhea, trauma, location of care, medications).

In addition, populate the `"diagnostic_test"` field as follows:

- If the clinical scenario explicitly **mentions a test**, populate that field accordingly.
- If **no test is mentioned**, but one is clearly implied (e.g., blood cultures for sepsis, ECG for chest pain), **infer** the most relevant diagnostic or monitoring test.
- If there is no reasonable test to mention, you may use `"N/A"` and leave the other fields blank.

---

**JSON Format:**

{{
  "patient_case": {{
  "prompt_question": "Paste the full clinical question text here.",
  "age": "Extract the patient’s age if given (e.g., '42-year-old')",
  "sex": "Extract the patient's sex (e.g., 'male', 'female')",
  "setting": "Mention the care setting if available (e.g., ED, ICU, outpatient, clinical trial).",
  "history": ["List relevant comorbidities, risk factors, or behaviors (e.g., IV drug use, recent surgery, travel)."],
  "symptoms": ["Extract main presenting symptoms (e.g., chest pain, shortness of breath, confusion)."],
  "vitals": ["Extract any vital signs (e.g., temperature, HR, BP, RR)."],
  "physical_exam": ["List key physical findings (e.g., murmur, rash, neuro deficit)."],
  "note": "Summarize the clinical focus of the question — diagnosis, management, or physiology."
  }},
  "diagnostic_test": {{
    "name": "Name the most relevant diagnostic or monitoring procedure if mentioned or clinically implied (e.g., blood culture, CT, echocardiogram, biopsy, telemetry).",
    "procedure": ["Briefly describe how the test would be performed."],
    "stains": ["If applicable (e.g., pathology, cytology), list stains."],
    "findings": ["What would be expected or observed in this patient."]
  }},
  "reasoning_tree": {{
    "start": "Review clinical case and question",
    "branch": [
      {{
        "condition": "Analyze each option based on known pathophysiology, presentation, and clues",
        "result": "Differential reasoning applied",
        "interpretation": "Compare each answer using specific features from the patient’s case",
        "diagnostic_options": [
          {{
            "option": "{options[0]}",
            "description": "...",
            "evidence_for": ["Mention how specific findings support this option."],
            "evidence_against": ["Mention how specific findings argue against it."],
            "final_verdict": "Most likely / Unlikely / Plausible"
          }},
          ...
        ]
      }}
    ],
    "conclusion": "Summarize why the selected answer is best given the case details."
  }}
}}

Respond with only the JSON object — no commentary or explanation outside the JSON.
"""

    try:
        client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
        )  # Set your API key
        # Generate response using GPT-4o
        response = client.chat.completions.create(
            model="o3-mini", # gpt-4o
            messages=[{"role": "user", "content": prompt}]
        )
        content = response.choices[0].message.content.strip()
        reasoning_json = json.loads(content)
        return reasoning_json

    except Exception as e:
        print(f"❌ Error processing question: {question[:80]}...\n{e}")
        return None


In [36]:
import openai
import json
import os
import time
from tqdm import tqdm

for idx, row in tqdm(df_op4.head(2).iterrows(), total=2):
    question = row['question']
    options = [row['opa'], row['opb'], row['opc'], row['opd']]
    
    reasoning_json = generate_reasoning_tree_json(question, options)
    
    if reasoning_json:
        filename = f"medbullet_q{idx+1:03d}.json"
        filepath = os.path.join(input_dir, filename)
        with open(filepath, 'w') as f:
            json.dump(reasoning_json, f, indent=2)
    
    time.sleep(1.2)  # throttle to stay within API limits

100%|█████████████████████████████████████████████| 2/2 [00:31<00:00, 15.65s/it]


In [37]:
import os
import json
from graphviz import Digraph
from tqdm import tqdm

# Input/output directories
input_dir = "MedBullet_Reasoning_Maps"
output_dir = "Reasoning_Map_Visualization"
os.makedirs(output_dir, exist_ok=True)

# Function to generate one flowchart from a JSON reasoning file
def render_reasoning_flowchart(filepath, output_png_path):
    with open(filepath, 'r') as f:
        data = json.load(f)

    dot = Digraph()
    dot.attr(rankdir='TB', fontsize='10')

    # ========== PATIENT CASE ==========
    case = data['patient_case']
    summary_parts = [
        f"Age: {case.get('age', 'N/A')}",
        f"Sex: {case.get('sex', 'N/A')}",
        f"Setting: {case.get('setting', 'N/A')}",
        "History:\n- " + '\n- '.join(case.get('history', [])) if case.get('history') else "History: N/A",
        "Symptoms:\n- " + '\n- '.join(case.get('symptoms', [])) if case.get('symptoms') else "Symptoms: N/A",
        "Vitals:\n- " + '\n- '.join(case.get('vitals', [])) if case.get('vitals') else "Vitals: N/A",
        "Physical Exam:\n- " + '\n- '.join(case.get('physical_exam', [])) if case.get('physical_exam') else "Physical Exam: N/A"
    ]
    dot.node('A1', f"Patient Summary:\n" + '\n'.join(summary_parts), shape='box')

    # ========== DIAGNOSTIC TEST ==========
    diag = data['diagnostic_test']
    procedure = '\n- ' + '\n- '.join(diag.get("procedure", [])) if diag.get("procedure") else "(none)"
    findings = '\n- ' + '\n- '.join(diag.get("findings", [])) if diag.get("findings") else "(none)"
    diag_node = f"Test:\n{diag.get('name', 'N/A')}\nProcedure:{procedure}\nFindings:{findings}"
    dot.node('B1', diag_node, shape='box')
    dot.edge('A1', 'B1', label='Perform diagnostic or monitoring test')

    # ========== INTERPRETATION ==========
    reasoning = data['reasoning_tree']
    branch = reasoning['branch'][0]
    dot.node('C1', f"Interpretation:\n{branch['interpretation']}", shape='box')
    dot.edge('B1', 'C1', label=branch['result'])

    # ========== OPTIONS ==========
    for i, option in enumerate(branch['diagnostic_options']):
        base_id = f"D{i}"
        dot.node(f"{base_id}a", f"{option['option']}", shape='box')
        dot.edge('C1', f"{base_id}a")

        dot.node(f"{base_id}b", option['description'], shape='note')
        dot.edge(f"{base_id}a", f"{base_id}b")

        last_node = f"{base_id}b"

        if option['evidence_for']:
            dot.node(f"{base_id}c", "✔ " + '\n✔ '.join(option['evidence_for']), shape='note')
            dot.edge(last_node, f"{base_id}c")
            last_node = f"{base_id}c"

        if option['evidence_against']:
            dot.node(f"{base_id}d", "✘ " + '\n✘ '.join(option['evidence_against']), shape='note')
            dot.edge(last_node, f"{base_id}d")
            last_node = f"{base_id}d"

        dot.node(f"{base_id}e", f"→ Verdict: {option['final_verdict']}", shape='box')
        dot.edge(last_node, f"{base_id}e")

        if option['final_verdict'].lower() == 'most likely':
            dot.edge(f"{base_id}e", 'Z')

    # ========== CONCLUSION ==========
    conclusion = reasoning['conclusion']
    dot.node('Z', f"Conclusion:\n{conclusion}", shape='box', style='filled', color='lightblue')

    # Save PNG output only
    dot.render(output_png_path, format='png', cleanup=True)

# ========== RUN FOR FIRST 2 JSON FILES ==========
json_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.json')])[:2]

for filename in tqdm(json_files):
    json_path = os.path.join(input_dir, filename)
    png_path = os.path.join(output_dir, filename.replace(".json", ""))
    render_reasoning_flowchart(json_path, png_path)

print(f"✅ Rendered flowcharts for the first 2 cases with full patient metadata.")


100%|█████████████████████████████████████████████| 2/2 [00:00<00:00,  2.29it/s]

✅ Rendered flowcharts for the first 2 cases with full patient metadata.
